# BOC baseline predictions — MM-ViT16 (run 1), multimodal (SWE image + kPa)

External clinical cohort (BOC, ketotifen study), 10 patients with RECIST 1.1 labels.
**One SWE image per patient** (no averaging), image + stiffness (kPa) → response / stable / non-response.

| Input | Source |
|---|---|
| Images | `02_data/artifact source/swe_patient_BOC/` — `p<N>_new.png` if it exists, else `p<N>.png` |
| True labels (RECIST 1.1) | `02_data/artifact source/NEW_DATA.xlsx`, column B (PR / SD / PD) |
| Patient kPa | `Export/SWE BOC/clinical table.xlsx`, column *Post-ketotifen stiffness (kPa)* |
| kPa rescaling statistics | all clinical SWE measurements in `Export/Original_excel_results ketotifen - April 2026 - FINAL.xlsx - data.csv`; mouse training kPa in `02_data/clustering_all_v5.xlsx` |
| Model | `03_models/MM-ViT16/run1/best_model_1.keras` (+ `best_model_image_only_1.keras` for the comparison panel) |
| Model kPa standardisation | `04_artifacts/MM-ViT16/run1/model_config.json` |

**kPa rescaling (label-free distribution matching).** Patient stiffness (C5-1 probe, deep human sarcomas, 3.8–20 kPa)
sits far below the murine training range (eL18-4 probe, 12.5–63.5 kPa). Each patient value is expressed relative to the
clinical cohort and placed on the murine scale:

$$\mathrm{kPa}_{\text{rescaled}} = \mu_{\text{mouse}} + \sigma_{\text{mouse}}\,\frac{\mathrm{kPa} - \mu_{\text{clinical}}}{\sigma_{\text{clinical}}}$$

No outcome labels are used to derive it. Requires TensorFlow / Keras **2.15** (the version the model was saved with).

In [ ]:
import json, os, re, tempfile, zipfile
from pathlib import Path
import numpy as np, pandas as pd, tensorflow as tf
from PIL import Image
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import confusion_matrix

# locate prognostic-ml root (folder containing 03_models) from the notebook location
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '03_models').is_dir())
EXPORT = ROOT.parent / 'Export'
OUT = Path.cwd()                      # results + figures are written next to this notebook
FIG = OUT / 'figures'; FIG.mkdir(exist_ok=True)
SRC = ROOT / '02_data' / 'artifact source'
CLASS_NAMES = ['response', 'stable', 'non-response']
print('ROOT  :', ROOT); print('EXPORT:', EXPORT); print('OUT   :', OUT); print('TF', tf.__version__)

## 1. Model loading
The `.keras` file must be used: the ViT16 positional embedding is a constant stored only in `config.json` (not in `.weights.h5`). Two small, output-identical patches let TF 2.15 deserialize it: the `cls_pick` Lambda → `TokenSlice`, and the constant `add_pos` input → an `AddConstant` layer holding the same values.

In [ ]:
class TokenSlice(layers.Layer):
    def __init__(self, index=0, **kw): super().__init__(**kw); self.index = index
    def call(self, x): return x[:, self.index, :]
    def get_config(self): return {**super().get_config(), 'index': self.index}

class CLSToken(layers.Layer):
    def __init__(self, dim, **kw): super().__init__(**kw); self.dim = dim
    def build(self, s): self.cls = self.add_weight(name='cls', shape=(1, 1, self.dim), initializer='random_normal')
    def call(self, x): return tf.concat([tf.tile(self.cls, [tf.shape(x)[0], 1, 1]), x], axis=1)
    def get_config(self): return {**super().get_config(), 'dim': self.dim}

class AddConstant(layers.Layer):
    def __init__(self, value, **kw):
        super().__init__(**kw); self.value = np.asarray(value, np.float32); self._c = tf.constant(self.value)
    def call(self, x): return x + self._c
    def get_config(self): return {**super().get_config(), 'value': self.value.tolist()}

CUSTOM = {'TokenSlice': TokenSlice, 'CLSToken': CLSToken, 'AddConstant': AddConstant}

def _patch(o):
    if isinstance(o, dict):
        if o.get('class_name') == 'Lambda' and o['config']['name'] == 'cls_pick':
            o.update(class_name='TokenSlice', module=None, registered_name='TokenSlice',
                     config={'name': 'cls_pick', 'trainable': True, 'dtype': 'float32', 'index': 0})
        if o.get('class_name') == 'Add' and '_CONSTANT_VALUE' in json.dumps(o.get('inbound_nodes', []))[:200]:
            node = o['inbound_nodes'][0]
            t = [e for e in node if e[0] != '_CONSTANT_VALUE']; c = [e for e in node if e[0] == '_CONSTANT_VALUE']
            o.update(class_name='AddConstant', module=None, registered_name='AddConstant',
                     config={'name': o['config']['name'], 'trainable': True, 'dtype': 'float32', 'value': c[0][2]},
                     inbound_nodes=[t]); o.pop('build_config', None)
        for v in o.values(): _patch(v)
    elif isinstance(o, list):
        for v in o: _patch(v)

def load_keras(path):
    with zipfile.ZipFile(path) as z:
        cfg = json.loads(z.read('config.json')); _patch(cfg)
        tmp = tempfile.NamedTemporaryFile(suffix='.keras', delete=False); tmp.close()
        with zipfile.ZipFile(tmp.name, 'w') as o:
            for n in z.namelist(): o.writestr(n, json.dumps(cfg) if n == 'config.json' else z.read(n))
    m = tf.keras.models.load_model(tmp.name, custom_objects=CUSTOM, compile=False, safe_mode=False)
    os.unlink(tmp.name); return m

MODEL_MM = ROOT / '03_models/MM-ViT16/run1/best_model_1.keras'
MODEL_IO = ROOT / '03_models/MM-ViT16/run1/best_model_image_only_1.keras'
mm, io = load_keras(MODEL_MM), load_keras(MODEL_IO)
cfg = json.load(open(ROOT / '04_artifacts/MM-ViT16/run1/model_config.json'))
print(f'multimodal params {mm.count_params():,}  (expected 315,011)')
print(f'image-only params {io.count_params():,}  (expected 314,883)')
print('model kPa standardisation: mean %.3f  SD %.3f' % (cfg['scaler_mean'], cfg['scaler_std']))

## 2. Cohort: images (one per patient), labels and kPa

In [ ]:
lab = pd.read_excel(SRC / 'NEW_DATA.xlsx', header=None, names=['study_id', 'recist', 'kpa_newdata'])
lab['pn'] = lab.study_id.str.extract(r'0*(\d+)')[0].astype(int)
lab['true'] = lab.recist.map({'PR': 'response', 'SD': 'stable', 'PD': 'non-response'})

ct = pd.read_excel(EXPORT / 'SWE BOC/clinical table.xlsx', header=None).iloc[24:39, [0, 5, 6]]
ct.columns = ['patient', 'kpa_baseline', 'kpa_post']
ct['pn'] = ct.patient.str.extract(r'(\d+)')[0].astype(int)

IMG = SRC / 'swe_patient_BOC'
df = lab.merge(ct[['pn', 'kpa_post']], on='pn')
df['image'] = [f'p{n}_new.png' if (IMG / f'p{n}_new.png').exists() else f'p{n}.png' for n in df.pn]
df[['study_id', 'recist', 'true', 'image', 'kpa_post']]

## 3. kPa rescaling (label-free)

In [ ]:
# clinical cohort: every SWE visit measurement (mean of the radiologist's readings) in the FINAL file
raw = pd.read_csv(EXPORT / 'Original_excel_results ketotifen - April 2026 - FINAL.xlsx - data.csv', header=None, dtype=str)
clin, pid = [], None
for _, r in raw.iterrows():
    m = re.match(r'0*(\d+)-Ket', str(r[0]).strip()); pid = int(m.group(1)) if m else pid
    d = pd.to_datetime(r[6], format='%m/%d/%Y', errors='coerce')
    v = [float(x) for x in re.findall(r'\d+\.?\d*', str(r[7]).replace('25.1.', '25.1,'))] if pd.notna(r[7]) else []
    if pid and pd.notna(d) and v: clin.append(np.mean(v))
clin = np.array(clin)
# murine training cohort: all 1,578 training images
mouse = pd.read_excel(ROOT / '02_data/clustering_all_v5.xlsx')['Elastic Modulus SWE (kPa)'].to_numpy(float)
MU_C, SD_C, MU_M, SD_M = clin.mean(), clin.std(), mouse.mean(), mouse.std()
rescale = lambda k: MU_M + SD_M * (np.asarray(k, float) - MU_C) / SD_C
print(f'clinical: n={len(clin)} mean {MU_C:.2f} SD {SD_C:.2f} (range {clin.min():.1f}-{clin.max():.1f})')
print(f'mouse   : n={len(mouse)} mean {MU_M:.2f} SD {SD_M:.2f} (range {mouse.min():.1f}-{mouse.max():.1f})')
print(f'=> kPa_rescaled = {MU_M - SD_M/SD_C*MU_C:.2f} + {SD_M/SD_C:.3f} x kPa')
df['kpa_rescaled'] = rescale(df.kpa_post)
df[['study_id', 'true', 'kpa_post', 'kpa_rescaled']].round(2)

## 4. Predictions (one image per patient, no averaging)

In [ ]:
def load_img(f):   # identical to the original BOC PNG inference: decode -> resize 304x400 -> /255
    x = tf.image.decode_png(tf.io.read_file(str(IMG / f)), channels=3)
    return tf.clip_by_value(tf.image.resize(x, [304, 400]) / 255.0, 0, 1).numpy()

X = np.stack([load_img(f) for f in df.image])
z = lambda k: ((np.asarray(k, float) - cfg['scaler_mean']) / cfg['scaler_std']).reshape(-1, 1).astype(np.float32)
P = mm.predict([X, z(df.kpa_rescaled)], verbose=0)
df[['p_response', 'p_stable', 'p_non_response']] = P
df['pred'] = np.array(CLASS_NAMES)[P.argmax(1)]
df['correct'] = df.pred == df.true

# comparison inputs (same images): raw clinical kPa, and the image-only model
df['pred_raw_kpa'] = np.array(CLASS_NAMES)[mm.predict([X, z(df.kpa_post)], verbose=0).argmax(1)]
df['pred_image_only'] = np.array(CLASS_NAMES)[io.predict(X, verbose=0).argmax(1)]

acc = {'Image + rescaled kPa': df.correct.sum(), 'Image + raw clinical kPa': (df.pred_raw_kpa == df.true).sum(),
       'Image only': (df.pred_image_only == df.true).sum()}
print({k: f'{v}/10' for k, v in acc.items()})
df.to_csv(OUT / 'baseline_predictions_multimodal.csv', index=False)
df[['study_id', 'image', 'true', 'kpa_post', 'kpa_rescaled', 'pred', 'correct', 'p_response', 'p_stable', 'p_non_response']].round(3)

## 5. Figure (Nature style; pastel; RECIST abbreviations PR / SD / PD)

In [ ]:
MM_ = 1 / 25.4
plt.rcParams.update({'font.family': 'Arial', 'font.size': 7, 'axes.labelsize': 7, 'axes.titlesize': 7,
    'xtick.labelsize': 6, 'ytick.labelsize': 6, 'legend.fontsize': 6, 'axes.linewidth': 0.5,
    'xtick.major.width': 0.5, 'ytick.major.width': 0.5, 'xtick.major.size': 2, 'ytick.major.size': 2,
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'axes.spines.top': False, 'axes.spines.right': False})
# RECIST 1.1 abbreviations; pastel palette
AB = {'response': 'PR', 'stable': 'SD', 'non-response': 'PD'}
PASTEL = {'response': '#9CC5E3', 'stable': '#F7D79C', 'non-response': '#EFA9A0'}
DARK = {'response': '#3A7DB8', 'stable': '#C9962B', 'non-response': '#C0574A'}
PCOLS = ['p_response', 'p_stable', 'p_non_response']

def letter(fig, ax, s, dx=0.06, dy=0.04):
    p = ax.get_position(); fig.text(p.x0 - dx, p.y1 + dy, s, fontsize=8, fontweight='bold', va='bottom')

def plot_cm(ax, true, pred, title):
    cm = confusion_matrix(true, pred, labels=CLASS_NAMES)
    ax.imshow(cm, cmap=LinearSegmentedColormap.from_list('b', ['#ffffff', '#2c5d8f']), vmin=0, vmax=max(cm.max(), 1))
    for i in range(3):
        for j in range(3):
            n, tot = cm[i, j], cm[i].sum()
            ax.text(j, i, f'{n}\n({100*n/tot:.0f}%)' if tot else '0', ha='center', va='center', fontsize=6.5,
                    color='white' if n > cm.max() / 2 else 'black')
    ax.set_xticks(range(3), [AB[c] for c in CLASS_NAMES]); ax.set_yticks(range(3), [AB[c] for c in CLASS_NAMES])
    ax.set_xlabel('Predicted'); ax.set_ylabel('True (RECIST 1.1)')
    for s in ax.spines.values(): s.set_visible(True)
    ok = int((np.asarray(true) == np.asarray(pred)).sum())
    ax.set_title(f'{title}\nAccuracy {ok}/{len(true)} ({100*ok/len(true):.0f}%)')

def plot_probs(ax, d, title, note_pn=None):
    d = d.assign(o=d.true.map({c: i for i, c in enumerate(CLASS_NAMES)})).sort_values(['o', 'pn']).reset_index(drop=True)
    x = np.arange(len(d)); b = np.zeros(len(d))
    for c, k in zip(CLASS_NAMES, PCOLS):
        ax.bar(x, d[k], bottom=b, color=PASTEL[c], width=0.72, lw=0.4, edgecolor='white', label=AB[c]); b += d[k].to_numpy()
    for i, r in d.iterrows():
        ax.text(i, 1.035, AB[r.true], ha='center', va='bottom', fontsize=6, color=DARK[r.true], fontweight='bold')
        if r.pred != r.true: ax.text(i, 1.11, '×', ha='center', va='bottom', fontsize=7)
    ax.text(-0.75, 1.035, 'True', ha='right', va='bottom', fontsize=6)
    ax.set_xticks(x, [f'P{n}' + ('†' if note_pn is not None and n == note_pn else '') for n in d.pn])
    ax.set_ylim(0, 1); ax.set_ylabel('Predicted probability'); ax.set_title(title, pad=24)
    ax.legend(title='Predicted', ncol=3, frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.13),
              handlelength=1, columnspacing=1.2, title_fontsize=6)


In [ ]:
for f in FIG.glob('*'): f.unlink()          # remove figures from earlier versions
fig = plt.figure(figsize=(183 * MM_, 62 * MM_))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 0.18, 2.1], wspace=0.35)
a = fig.add_subplot(gs[0, 0]); plot_cm(a, df.true, df.pred, 'Baseline · image + kPa')
b = fig.add_subplot(gs[0, 2]); plot_probs(b, df, 'Baseline · per-patient probability')
letter(fig, a, 'a', 0.07); letter(fig, b, 'b', 0.05)
fig.savefig(FIG / 'Figure_baseline_multimodal.pdf', bbox_inches='tight'); fig.savefig(FIG / 'Figure_baseline_multimodal.png', dpi=600, bbox_inches='tight')
plt.show(); print('Saved to', FIG); print(sorted(p.name for p in FIG.iterdir()))